# CS781 Phase 1 — Model 2 (MLA + DSA + MoE) — Training

**Inputs to attach** (Add Data): (1) the tokenizer dataset, (2) the output of the tokenization notebook (`shards/`, `val_tokens.npy`, `tokenized_meta.json`), and, from the second session on, (3) the previous version of *this* notebook's output.

Reuses the SAME tokenized shards as model 1 -- they're flat token arrays, not pre-blocked, so the 4096-token context here just reshapes them differently; no re-tokenizing needed.

**Per run you only change `STEPS_THIS_RUN` (and `TOKEN_BUDGET`/`PER_DEVICE_BATCH` while calibrating).** This architecture (MLA latent attention + DSA sparse selection + MoE) has a very different memory/speed profile from model 1's GQA -- treat the batch size below as a starting guess to be corrected by the calibration run, not a known-good value.

Needs no internet and no Hugging Face login.

## 1. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
N_GPU = max(1, torch.cuda.device_count())

## 2. Configuration

In [ ]:
import os, json, glob, time, math, shutil
import numpy as np

SESSION_START = time.time()

# ===== change these per run =====
STEPS_THIS_RUN = None      # optimizer steps for THIS session. 2 = quick timing/diagnostic check. None = run until the
                        # time limit below (or the end of training), whichever comes first
TOKEN_BUDGET = 950_000_000  # training tokens to use. Calibration: small. Real run: set from the printed suggestion
MAX_SESSION_HOURS = 4.5  # training stops + checkpoints after this long, so one commit always ends
                          # cleanly under the 12h limit (leaves time for final eval + save)

# ===== fixed for the whole training: do NOT change between sessions =====
SEED = 42
BLOCK = 4096                      # training sequence length (tokens) -- 4x model 1's, chosen so
                                  # NSA/DSA-style sparsity actually has something to save on
N_EMBD, N_LAYER, N_HEAD, HEAD_DIM = 1024, 8, 16, 64
D_C, D_C_Q, D_ROTATE = 128, 192, 32                                    # MLA latent dims
N_IDX_HEADS, IDX_HEAD_DIM, CHUNK_SIZE = 8, 32, 16                       # DSA indexer
TOPK_CHUNKS, FORCED_LOCAL_CHUNKS, Q_BLOCK = 64, 2, 512                  # DSA selection (64*16=1024 tokens, ~25% of 4096)
N_ROUTED_EXPERTS, N_SHARED_EXPERTS, MOE_TOP_K, EXPERT_D_FF = 6, 1, 2, 896
INDEXER_LOSS_COEF = 0.1
INDEXER_DISTILL_SAMPLE = 256              # query positions the distillation target is computed over per step,
                                           # not all 4096 -- that was an uncounted O(T^2) cost that dominated
                                           # real training time (measured ~10x+ speedup from this alone)

# starting guess only -- this architecture's memory profile is untested; the calibration
# run's diagnostics (peak memory, OOM-or-not) are what actually sets this, not this number
PER_DEVICE_BATCH = 2
GRAD_ACCUM_STEPS = 128           # 2 seqs x 2 GPUs x 128 = 512 seqs/step, same global batch as model 1
LEARNING_RATE = 3e-4
WARMUP_STEPS = 50
SAVE_STEPS = 100                  # checkpoint every N optimizer steps
EVAL_STEPS = 200

# ---- data mix (fixed for the whole training; SAME source doc counts as model 1) ----
SOURCES = ["acts", "high_courts", "supreme_court", "tribunals"]   # the order the tokenization notebook read the files in
SOURCE_DOCS = {"acts": 18_826, "high_courts": 2_536_738, "supreme_court": 55_549, "tribunals": 869_132}
MIX = "equal"                     # "equal" | "proportional" | {"acts": 1, "high_courts": 2, ...} (relative weights)
VAL_RUNS, VAL_RUN_BLOCKS, VAL_MARGIN_BLOCKS = 25, 10, 8   # per source: 25 runs x 10 blocks held out, margin kept out of training
TOTAL_GPU_HOUR_BUDGET = 30.0
TRAIN_BUDGET_HOURS = 28.0         # plan the whole run to fit this, leaving slack under the 30h

OUTPUT_DIR = "/kaggle/working/cs781_phase1_model2"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-mladsa-final")
TRAINING_META_PATH = os.path.join(OUTPUT_DIR, "training_meta.json")
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

## 3. Load Tokenizer and Token Shards

Same tokenizer and same token shards as model 1 (the tokenization notebook's output) -- attach both the tokenizer dataset and that notebook's output here, same as before. Inputs are located by searching `/kaggle/input`, so the exact mount path doesn't matter.

In [ ]:
from transformers import AutoTokenizer

def show_inputs():
    print("Attached inputs under /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        print("  " * depth + (os.path.basename(root) or "input") + "/", f"({len(files)} files)" if files else "")
        if depth >= 3:
            dirs[:] = []

def find_dirs(filename):
    return sorted({os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)})

# tokenizer.json also appears inside this notebook's own saved checkpoints; ignore those
cands = [d for d in find_dirs("tokenizer.json") if "checkpoint-" not in d and "legal-mladsa-final" not in d]
if not cands:
    show_inputs()
    raise FileNotFoundError("No tokenizer found. Attach your tokenizer DATASET to this notebook (Add Input); "
                            "the listing above shows what is currently attached.")
TOKENIZER_DIR = ([d for d in cands if os.path.exists(os.path.join(d, "trained_on.json"))] or cands)[0]
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
print("Tokenizer:", TOKENIZER_DIR, "| vocab size:", len(tokenizer))

meta_dirs = find_dirs("tokenized_meta.json")
if not meta_dirs:
    show_inputs()
    raise FileNotFoundError("No tokenized_meta.json found. Attach the OUTPUT of your tokenization notebook to this notebook.")
TOKEN_ROOT = meta_dirs[0]
META_PATH = os.path.join(TOKEN_ROOT, "tokenized_meta.json")
meta = json.load(open(META_PATH))
assert meta["vocab_size"] == len(tokenizer), "token shards were built with a different tokenizer!"
print(f"Token data: {TOKEN_ROOT} | {meta['train_tokens']:,} train tokens in {len(meta['shards'])} shards")

## 3b. Build the Training Mix and Validation Set

Identical logic to model 1's mix-building cell (same corpus, same equal-mixture-per-source approach) -- only `BLOCK` differs (4096 here vs 1024 there), which just changes how the SAME flat token shards get reshaped into sequences.

In [ ]:
import torch

# ---- 1. open every shard (memory-mapped: nothing is loaded into RAM) ----
arrays = [np.load(os.path.join(TOKEN_ROOT, "shards", s), mmap_mode="r") for s in meta["shards"]]
eos = tokenizer.eos_token_id
shard_len = np.array([len(a) for a in arrays])
tok_off = np.concatenate([[0], np.cumsum(shard_len)])          # global token offset where each shard starts
n_blk = shard_len // BLOCK
cum_blk = np.concatenate([[0], np.cumsum(n_blk)])              # global id of each shard's first block
N_BLOCKS = int(cum_blk[-1])

# ---- 2. where does each source start? (count end-of-document tokens) ----
t_scan = time.time()
assert sum(SOURCE_DOCS.values()) == meta["num_docs"], "SOURCE_DOCS doesn't add up to the tokenized corpus"
val_arr = np.load(os.path.join(TOKEN_ROOT, "val_tokens.npy"))
n_val_docs = int((val_arr == eos).sum()) + (0 if int(val_arr[-1]) == eos else 1)   # tokenization sent every 20th doc to val until full
eos_per_shard = np.array([int(np.count_nonzero(a == eos)) for a in arrays])
cum_eos = np.concatenate([[0], np.cumsum(eos_per_shard)])
assert int(cum_eos[-1]) == meta["num_docs"] - n_val_docs, (
    f"doc count mismatch: {int(cum_eos[-1]):,} end-of-doc tokens in train shards vs "
    f"{meta['num_docs']:,} docs - {n_val_docs:,} val docs. The token data isn't what this cell assumes.")

def train_doc_index(g):
    # global doc index -> index among train docs (val docs were global indices 0, 20, 40, ... n_val_docs of them)
    return g - min(n_val_docs, -(-g // 20))

def token_offset_of_train_doc(t):
    # global token offset where the t-th train doc starts = just after the t-th end-of-document token
    if t == 0:
        return 0
    j = int(np.searchsorted(cum_eos, t, side="left")) - 1
    pos = np.flatnonzero(arrays[j] == eos)
    return int(tok_off[j] + pos[t - int(cum_eos[j]) - 1] + 1)

doc_edges = np.cumsum([0] + [SOURCE_DOCS[s] for s in SOURCES])
src_start_tok = np.array([0] + [token_offset_of_train_doc(train_doc_index(int(g))) for g in doc_edges[1:-1]])

# ---- 3. which source does every block belong to? ----
shard_of_blk = np.repeat(np.arange(len(arrays)), n_blk)
blk_start_tok = tok_off[shard_of_blk] + (np.arange(N_BLOCKS) - cum_blk[shard_of_blk]) * BLOCK
blk_source = np.searchsorted(src_start_tok[1:], blk_start_tok, side="right")     # 0..len(SOURCES)-1, non-decreasing
src_lo = np.searchsorted(blk_source, np.arange(len(SOURCES)), side="left")
src_hi = np.searchsorted(blk_source, np.arange(len(SOURCES)), side="right")

# ---- 4. hold out validation: evenly spread random runs per source, with a margin excluded from training ----
rng = np.random.default_rng(SEED)
excluded = np.zeros(N_BLOCKS, dtype=bool)
val_ids = {}
for si, s in enumerate(SOURCES):
    lo, hi = int(src_lo[si]), int(src_hi[si])
    stratum = (hi - lo) // VAL_RUNS
    assert stratum > VAL_RUN_BLOCKS + 2 * VAL_MARGIN_BLOCKS + 1, f"source {s} is too small for {VAL_RUNS} validation runs"
    runs = []
    for r in range(VAL_RUNS):
        a = lo + r * stratum + VAL_MARGIN_BLOCKS + int(rng.integers(0, stratum - VAL_RUN_BLOCKS - 2 * VAL_MARGIN_BLOCKS))
        runs.append(np.arange(a, a + VAL_RUN_BLOCKS))
        excluded[a - VAL_MARGIN_BLOCKS: a + VAL_RUN_BLOCKS + VAL_MARGIN_BLOCKS] = True
    val_ids[s] = np.concatenate(runs)

# ---- 5. how many blocks does each source contribute? ----
cand = {s: np.arange(src_lo[i], src_hi[i])[~excluded[src_lo[i]:src_hi[i]]] for i, s in enumerate(SOURCES)}
avail = {s: len(cand[s]) for s in SOURCES}

def allocate(avail, weights, total):
    # split `total` blocks across sources in proportion to `weights`, never exceeding what a source has;
    # whatever a small source can't absorb is redistributed over the others
    alloc = {s: 0 for s in avail}
    active = [s for s in avail if weights[s] > 0]
    remaining = int(total)
    while active and remaining > 0:
        wsum = sum(weights[s] for s in active)
        share = {s: remaining * weights[s] / wsum for s in active}
        capped = [s for s in active if share[s] >= avail[s]]
        if not capped:
            for s in active:
                alloc[s] = int(share[s])
            break
        for s in capped:
            alloc[s] = avail[s]
            remaining -= avail[s]
            active.remove(s)
    return alloc

if MIX == "equal":
    weights = {s: 1.0 for s in SOURCES}
elif MIX == "proportional":
    weights = {s: float(avail[s]) for s in SOURCES}
else:
    weights = {s: float(MIX[s]) for s in SOURCES}
alloc = allocate(avail, weights, TOKEN_BUDGET // BLOCK)
train_ids = rng.permutation(np.concatenate([rng.choice(cand[s], size=alloc[s], replace=False) for s in SOURCES]))

# ---- 6. datasets ----
class PickedBlocks(torch.utils.data.Dataset):
    """Fixed-length blocks, read on demand from the memory-mapped shards by global block id."""
    def __init__(self, ids):
        self.ids = np.asarray(ids)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        g = int(self.ids[i])
        j = int(np.searchsorted(cum_blk, g, side="right") - 1)
        k = g - int(cum_blk[j])
        return {"input_ids": arrays[j][k * BLOCK:(k + 1) * BLOCK].astype(np.int64).tolist()}

train_dataset = PickedBlocks(train_ids)
val_by_source = {s: PickedBlocks(val_ids[s]) for s in SOURCES}
val_dataset = PickedBlocks(np.concatenate([val_ids[s] for s in SOURCES]))

GLOBAL_SEQS = PER_DEVICE_BATCH * N_GPU * GRAD_ACCUM_STEPS
TOTAL_TRAIN_STEPS = len(train_dataset) // GLOBAL_SEQS          # one pass over the data

print(f"scanned {len(arrays)} shards in {time.time() - t_scan:.0f}s | {meta['num_docs']:,} docs, {n_val_docs:,} of them were held out by the tokenizer")
print(f"MIX = {MIX!r} | budget {TOKEN_BUDGET/1e6:,.0f}M tokens (corpus has {N_BLOCKS * BLOCK / 1e9:.2f}B)")
print(f"{'source':14s}{'available':>11s}{'used':>10s}{'share':>8s}{'of source':>11s}")
_tot = max(sum(alloc.values()), 1)
for s in SOURCES:
    print(f"{s:14s}{avail[s] * BLOCK / 1e6:10.0f}M{alloc[s] * BLOCK / 1e6:9.0f}M{100 * alloc[s] / _tot:7.1f}%{100 * alloc[s] / max(avail[s], 1):10.1f}%")
print(f"validation: {len(val_dataset):,} blocks ({VAL_RUNS * VAL_RUN_BLOCKS} per source, in {VAL_RUNS} runs of {VAL_RUN_BLOCKS})")
print(f"Train blocks: {len(train_dataset):,} | Tokens/step: {GLOBAL_SEQS * BLOCK:,} | TOTAL_TRAIN_STEPS (one epoch): {TOTAL_TRAIN_STEPS:,}")
if TOKEN_BUDGET < 1_000_000_000:
    print("NOTE: small TOKEN_BUDGET -> calibration only. Set it from the printed suggestion for the real run.")

## 4. Define The Model

Five files this time, not one: `mla.py` (Multi-head Latent Attention), `dsa.py` (the lightning indexer + chunked sparse selection), `moe.py` (routed + shared experts), `mla_dsa.py` (combines the first two into one attention module), `model.py` (the decoder stack). Written to real files and imported for the same reason as model 1: `transformers` reads `__file__` off a model class's module, and a class defined live in a notebook cell has none.

Every piece here (causal safety across an exhaustive per-position sweep, the block-shared-selection causality fix, the padding-boundary fix for incremental decode, the indexer's KL-distillation loss, checkpointing equivalence, tied-embedding save/reload) was tested in a sandbox before being written here -- see the build conversation for the specific bugs that testing caught.

In [ ]:
%%writefile /kaggle/working/mla.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    # x: (..., T, d); cos/sin: (T, d)
    return (x * cos) + (rotate_half(x) * sin)


def rope_table(T, dim, theta, device, dtype):
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    pos = torch.arange(T, device=device).float()
    emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
    return emb.cos().to(dtype), emb.sin().to(dtype)


class MLA(nn.Module):
    """Multi-head Latent Attention (DeepSeek-V2 style).
    Caches only the compressed latent c_kv + the shared decoupled RoPE key k_R,
    not full per-head K/V. head_dim is the per-head size of the UP-PROJECTED
    (decompressed) content part; d_rotate is appended on top of that for the
    RoPE-carrying part, so the actual attention dim per head is head_dim+d_rotate.
    """
    def __init__(self, d_model, n_head, head_dim, d_c, d_c_q, d_rotate, dropout=0.0, rope_theta=10000.0):
        super().__init__()
        self.n_head, self.head_dim, self.d_rotate = n_head, head_dim, d_rotate
        self.rope_theta = rope_theta
        self.dropout_p = dropout

        self.W_DKV = nn.Linear(d_model, d_c, bias=False)
        self.W_UK = nn.Linear(d_c, n_head * head_dim, bias=False)
        self.W_UV = nn.Linear(d_c, n_head * head_dim, bias=False)
        self.W_KR = nn.Linear(d_model, d_rotate, bias=False)  # shared across heads

        self.W_DQ = nn.Linear(d_model, d_c_q, bias=False)
        self.W_UQ = nn.Linear(d_c_q, n_head * head_dim, bias=False)
        self.W_QR = nn.Linear(d_c_q, n_head * d_rotate, bias=False)  # per head

        self.W_O = nn.Linear(n_head * head_dim, d_model, bias=False)

    def project(self, x):
        """Splits the model into the compressed-latent step (cacheable) and the
        up-projection + attention step, so DSA can sit between them and score
        against the SAME K^C/k_R this attention will use."""
        B, T, _ = x.shape
        c_kv = self.W_DKV(x)                          # (B, T, d_c)  <- what gets cached
        c_q = self.W_DQ(x)                             # (B, T, d_c_q)
        k_R = self.W_KR(x)                              # (B, T, d_rotate) <- also cached, shared across heads
        return c_kv, c_q, k_R

    def up_project(self, c_kv, c_q, k_R, past_len, device, dtype):
        B, T, _ = c_kv.shape
        H, Dh, Dr = self.n_head, self.head_dim, self.d_rotate

        K_C = self.W_UK(c_kv).view(B, T, H, Dh).transpose(1, 2)   # (B,H,T,Dh)
        V_C = self.W_UV(c_kv).view(B, T, H, Dh).transpose(1, 2)   # (B,H,T,Dh)
        Q_C = self.W_UQ(c_q).view(B, T, H, Dh).transpose(1, 2)    # (B,H,T,Dh)
        Q_R = self.W_QR(c_q).view(B, T, H, Dr).transpose(1, 2)    # (B,H,T,Dr)

        # RoPE table dtype comes from Q_R itself (the tensor actually being rotated),
        # NOT the passed-in `dtype` -- that's the pre-projection RMSNorm output, which
        # autocast can leave in fp32 (reductions like RMSNorm's .mean() are kept fp32
        # for stability) even while W_QR (a Linear/matmul) is correctly forced to fp16.
        # Using the stale pre-projection dtype here builds an fp32 cos/sin table, and
        # multiplying that into a correctly-fp16 Q_R silently promotes it back to fp32
        # -- exactly the dtype drift that caused indexer_distillation_loss's crash
        # downstream (Q ending up fp32 while indexer_scores, built the "right" way from
        # a post-projection dtype, correctly stayed fp16).
        cos, sin = rope_table(past_len + T, Dr, self.rope_theta, device, Q_R.dtype)
        cos, sin = cos[past_len:past_len + T], sin[past_len:past_len + T]
        Q_R = apply_rope(Q_R, cos, sin)
        k_R_rot = apply_rope(k_R.to(Q_R.dtype), cos, sin)          # (B,T,Dr)
        K_R = k_R_rot.unsqueeze(1).expand(B, H, T, Dr)             # broadcast to all heads

        Q = torch.cat([Q_C, Q_R], dim=-1)   # (B,H,T,Dh+Dr)
        K = torch.cat([K_C, K_R], dim=-1)   # (B,H,T,Dh+Dr)
        return Q, K, V_C

    def forward(self, x, past_kv=None, use_cache=False, key_padding_mask=None):
        """past_kv, if given: (c_kv_past, k_R_past) each (B, T_past, ·) — the
        compact cached state. NOT full per-head K/V."""
        B, T, _ = x.shape
        past_len = past_kv[0].shape[1] if past_kv is not None else 0

        c_kv, c_q, k_R = self.project(x)
        if past_kv is not None:
            full_c_kv = torch.cat([past_kv[0], c_kv], dim=1)
            full_k_R = torch.cat([past_kv[1], k_R], dim=1)
        else:
            full_c_kv, full_k_R = c_kv, k_R

        # Q only ever needed for the T new positions; K/V up-projected from the FULL cached latent
        Q, _, _ = self.up_project(c_kv, c_q, k_R, past_len, x.device, x.dtype)
        _, K_full, V_full = self._up_project_kv_only(full_c_kv, full_k_R, x.device, x.dtype)

        total_len = past_len + T
        if key_padding_mask is None and past_kv is None:
            attn_mask, is_causal = None, True
        else:
            # dtype from Q (post-projection, correctly typed after the up_project fix
            # above), not x (pre-projection, which autocast can leave in fp32) -- same
            # class of mismatch that crashed indexer_distillation_loss, fixed here too
            # since this path (padding / cached generation) wasn't exercised by the
            # training run that caught the original crash and could hit it later.
            q_pos = torch.arange(past_len, past_len + T, device=x.device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=x.device).unsqueeze(0)
            causal = (k_pos > q_pos).to(Q.dtype) * torch.finfo(Q.dtype).min
            bias = causal[None, None]
            if key_padding_mask is not None:
                pad = (1.0 - key_padding_mask[:, None, None, :].to(Q.dtype)) * torch.finfo(Q.dtype).min
                bias = bias + pad
            attn_mask, is_causal = bias, False

        out = F.scaled_dot_product_attention(
            Q, K_full, V_full, attn_mask=attn_mask, is_causal=is_causal,
            dropout_p=self.dropout_p if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        out = self.W_O(out)
        present = (full_c_kv, full_k_R) if use_cache else None
        return out, present

    def _up_project_kv_only(self, c_kv, k_R, device, dtype):
        B, T, _ = c_kv.shape
        H, Dh, Dr = self.n_head, self.head_dim, self.d_rotate
        K_C = self.W_UK(c_kv).view(B, T, H, Dh).transpose(1, 2)
        V_C = self.W_UV(c_kv).view(B, T, H, Dh).transpose(1, 2)
        # same fix as up_project: derive RoPE dtype from a post-projection tensor
        # (K_C, correctly fp16 under autocast), not the passed-in pre-projection dtype
        cos, sin = rope_table(T, Dr, self.rope_theta, device, K_C.dtype)
        k_R_rot = apply_rope(k_R.to(K_C.dtype), cos, sin)
        K_R = k_R_rot.unsqueeze(1).expand(B, H, T, Dr)
        return None, torch.cat([K_C, K_R], dim=-1), V_C

In [ ]:
%%writefile /kaggle/working/dsa.py
import torch
import torch.nn as nn
import torch.nn.functional as F

from mla import rope_table, apply_rope


class Indexer(nn.Module):
    """Lightning indexer: cheap per-CHUNK relevance scores, feeding DSA's chunk
    selection. Keys are pooled to one summary vector per chunk BEFORE scoring
    (not scored per-token and pooled after) -- the indexer's decision is at
    chunk granularity regardless, so per-token precision here is pure waste:
    it costs O(T_block x T) for a decision that's O(T_block x num_chunks).
    Queries come from MLA's own query latent c_q (no extra projection from
    raw x); keys derive from MLA's cached c_kv; both reuse MLA's RoPE table."""
    def __init__(self, d_c_q, d_c, n_idx_heads, idx_head_dim, chunk_size, rope_theta=10000.0):
        super().__init__()
        self.H, self.Dh, self.chunk_size = n_idx_heads, idx_head_dim, chunk_size
        self.rope_theta = rope_theta
        self.W_IQ = nn.Linear(d_c_q, n_idx_heads * idx_head_dim, bias=False)
        self.W_IK = nn.Linear(d_c, idx_head_dim, bias=False)          # single shared "head"
        self.W_IW = nn.Linear(d_c_q, n_idx_heads, bias=False)          # per-query head weights

    def forward(self, c_q, c_kv, past_len=0):
        B, T, _ = c_q.shape
        Tk = c_kv.shape[1]
        cs = self.chunk_size
        num_chunks = (Tk + cs - 1) // cs
        pad = num_chunks * cs - Tk

        q = self.W_IQ(c_q).view(B, T, self.H, self.Dh)
        k = self.W_IK(c_kv)                                            # (B, Tk, Dh)
        w = self.W_IW(c_q)                                              # (B, T, H)

        cos, sin = rope_table(Tk, self.Dh, self.rope_theta, c_q.device, c_q.dtype)
        k = apply_rope(k, cos, sin)                                     # RoPE per real token position, same as before
        q = apply_rope(q.transpose(1, 2), cos[past_len:past_len + T], sin[past_len:past_len + T]).transpose(1, 2)

        if pad:
            k = F.pad(k, (0, 0, 0, pad))                                # zero-pad tail to a full chunk
        k_pooled = k.view(B, num_chunks, cs, self.Dh).mean(dim=2)       # (B, num_chunks, Dh) -- pool AFTER RoPE,
                                                                          # BEFORE scoring: the indexer only ever
                                                                          # needs a chunk-level relevance signal

        # I[b,t,c] = sum_h w[b,t,h] * relu(q[b,t,h,:] . k_pooled[b,c,:])
        scores = torch.einsum("bthd,bcd->bhtc", q, k_pooled)           # (B,H,T,num_chunks)
        scores = F.relu(scores)
        scores = torch.einsum("bhtc,bth->btc", scores, w)              # (B,T,num_chunks) -- ALREADY chunk-level
        return scores


def indexer_distillation_loss(Q, K, indexer_scores, chunk_size, causal_offset=0, sample_size=256):
    """The indexer's scores only ever feed a hard top-k, which is
    non-differentiable -- with nothing else, it never receives a training
    signal at all. DeepSeek's own recipe trains the indexer separately, via
    KL-divergence against the real (dense) attention distribution, computed
    here from Q/K already on hand. Only the QK^T + softmax is recomputed
    (no V-multiply, no backward through it -- it's a detached target).

    Computed over a random SUBSAMPLE of `sample_size` query positions, not
    all T -- this target is itself an O(T^2) dense computation (materializing
    a (B,H,T,T) tensor), and left at full T it dominates real training time
    (confirmed: ~16x the cost measured at T=1024, scaling quadratically,
    every layer, every step -- an OOM even on CPU at the real T=4096).
    Every query still gets sampled roughly every T/sample_size steps on
    average across training, which is enough for a KL-distillation signal
    that only needs to shape the indexer's general relevance judgment, not
    match it exactly at every position on every single step.

    indexer_scores is now chunk-level (B,T,num_chunks), so the token-level
    dense-attention target is pooled to the same granularity by SUMMING
    (not averaging) each chunk's token probabilities -- the correct way to
    marginalize a probability distribution over grouped outcomes. Both sides
    are then restricted to the same "whole chunk causally available" support
    that chunked_sparse_attention itself uses (a chunk counts as available
    only once its END position is valid), so the indexer is trained toward
    the same notion of availability that selection actually applies."""
    B, H, T, Dqk = Q.shape
    Tk = K.shape[2]
    device, dtype = Q.device, Q.dtype
    num_chunks = (Tk + chunk_size - 1) // chunk_size
    pad = num_chunks * chunk_size - Tk

    if T > sample_size:
        sample_idx = torch.randperm(T, device=device)[:sample_size]
        Qs = Q[:, :, sample_idx, :]
        scores_s = indexer_scores[:, sample_idx, :]
        q_abs = causal_offset + sample_idx
    else:
        Qs = Q
        scores_s = indexer_scores
        q_abs = torch.arange(causal_offset, causal_offset + T, device=device)
    Ts = Qs.shape[2]

    q_pos = q_abs.unsqueeze(1)
    k_pos = torch.arange(0, Tk, device=device).unsqueeze(0)
    causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
    with torch.no_grad():
        target_logits = torch.einsum("bhtd,bhsd->bhts", Qs, K) / (Dqk ** 0.5) + causal[None, None]
        target_tok = F.softmax(target_logits, dim=-1).mean(dim=1)          # (B,Ts,Tk), averaged over heads
        if pad:
            target_tok = F.pad(target_tok, (0, pad), value=0.0)
        target = target_tok.view(B, Ts, num_chunks, chunk_size).sum(dim=-1)  # (B,Ts,num_chunks)

    real_chunk_start_pos = torch.arange(num_chunks, device=device) * chunk_size
    # A chunk only needs to START on-or-before the query to count as "available" here
    # (unlike chunked_sparse_attention's stricter end-position rule, which exists only
    # to guarantee a whole gathered chunk is safe -- this loss has no gather to protect).
    # Using that stricter rule here would mark EVERY chunk unavailable for the earliest
    # queries (even chunk 0, whose END is still in the future for query 0), giving an
    # all-(-inf) row and a NaN from log_softmax -- exactly what testing caught.
    chunk_not_ok = (real_chunk_start_pos[None, :] > q_pos)                  # (Ts,num_chunks)
    idx_logits = scores_s.masked_fill(chunk_not_ok[None], torch.finfo(scores_s.dtype).min)
    # NOTE: torch.finfo(dtype).min here, not float("-inf") -- log_softmax of a masked
    # position legitimately produces -inf, and F.kl_div computes target*input BEFORE
    # summing, so a real -inf meeting target=0 at that same position gives 0*(-inf) = NaN
    # in IEEE float, regardless of the mathematical convention that it "should" be 0.
    # A large finite negative number sidesteps this (same pattern already used for the
    # safety mask in chunked_sparse_attention, for the same underlying reason).
    idx_log_probs = F.log_softmax(idx_logits, dim=-1)

    target = target.masked_fill(chunk_not_ok[None], 0.0)
    target = target / target.sum(dim=-1, keepdim=True).clamp_min(1e-8)      # renormalize over the same support
    return F.kl_div(idx_log_probs, target, reduction="batchmean")


def chunked_sparse_attention(Q, K, V, indexer_scores, chunk_size, topk_chunks,
                              forced_local_chunks, q_block, causal_offset=0):
    """Q,K,V: (B,H,T,·). indexer_scores: (B,Tq,num_chunks) -- ALREADY chunk-level
    (the Indexer pools keys to chunk summaries before scoring, so there's no
    token-level score matrix to pool here anymore). Selection is shared across
    each block of q_block consecutive queries (see memory note above) -- attention
    itself still applies the exact per-position causal mask, so a chunk that slips
    into an early query's selected set but lies in that query's future is harmless,
    just wasted capacity, never a correctness issue."""
    B, H, Tq, Dqk = Q.shape
    Tk = K.shape[2]
    device, dtype = Q.device, Q.dtype
    num_chunks = (Tk + chunk_size - 1) // chunk_size
    pad = num_chunks * chunk_size - Tk
    if pad:
        K = F.pad(K, (0, 0, 0, pad))
        V = F.pad(V, (0, 0, 0, pad))
    K = K.view(B, H, num_chunks, chunk_size, Dqk)
    V = V.view(B, H, num_chunks, chunk_size, V.shape[-1])

    assert indexer_scores.shape[-1] == num_chunks, \
        f"indexer_scores last dim ({indexer_scores.shape[-1]}) must equal num_chunks ({num_chunks})"
    chunk_scores = indexer_scores  # already chunk-level, nothing to pool

    outputs = []
    for q0 in range(0, Tq, q_block):
        q1 = min(q0 + q_block, Tq)
        blk = q1 - q0
        last_pos = causal_offset + q1 - 1          # most-permissive causal bound for this block
        # A chunk's REAL end may be less than its padded/nominal end (the last chunk is
        # often only partially filled, e.g. mid-generation with a cache length that
        # isn't a multiple of chunk_size). Using the padded end here would wrongly mark
        # a chunk "not yet causally available" even when it holds the current token --
        # exactly the source of the incremental-vs-full-forward mismatch found in testing.
        real_chunk_end_pos = torch.clamp(
            torch.arange(num_chunks, device=device) * chunk_size + (chunk_size - 1), max=Tk - 1
        )
        causal_ok = real_chunk_end_pos <= last_pos
        # Selection must depend ONLY on the block's earliest (most restrictive) query --
        # pooling over the whole block would let a later query's data (which the
        # earlier queries can't legally see yet) influence what gets selected for them,
        # a real causality leak even though the final attention weights stay masked.
        block_score = chunk_scores[:, q0].clone()                            # (B,num_chunks)
        block_score = block_score.masked_fill(~causal_ok, float("-inf"))

        this_last_chunk = last_pos // chunk_size
        forced = torch.arange(max(0, this_last_chunk - forced_local_chunks + 1),
                               this_last_chunk + 1, device=device)
        block_score = block_score.clone()
        block_score[:, forced] = float("inf")                              # guarantee local window is kept

        k_sel = min(topk_chunks, int(causal_ok.sum().item()) if causal_ok.any() else 1)
        sel = block_score.topk(k_sel, dim=-1).indices                       # (B, k_sel)

        Dv = V.shape[-1]
        gK = torch.gather(K, 2, sel[:, None, :, None, None].expand(B, H, k_sel, chunk_size, Dqk))
        gK = gK.reshape(B, H, k_sel * chunk_size, Dqk)
        gV = torch.gather(V, 2, sel[:, None, :, None, None].expand(B, H, k_sel, chunk_size, Dv))
        gV = gV.reshape(B, H, k_sel * chunk_size, Dv)

        gathered_pos = (sel[:, :, None] * chunk_size + torch.arange(chunk_size, device=device)[None, None, :])
        gathered_pos = gathered_pos.reshape(B, k_sel * chunk_size)          # (B, gathered_len)
        q_pos = torch.arange(q0, q1, device=device) + causal_offset          # (blk,)
        safety_mask = (gathered_pos[:, None, :] > q_pos[None, :, None])     # (B, blk, gathered_len)
        bias = safety_mask[:, None, :, :].to(dtype) * torch.finfo(dtype).min

        Qb = Q[:, :, q0:q1]
        out = F.scaled_dot_product_attention(Qb, gK, gV, attn_mask=bias)
        outputs.append(out)
    return torch.cat(outputs, dim=2)

In [ ]:
%%writefile /kaggle/working/moe.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class Expert(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class MoE(nn.Module):
    def __init__(self, d_model, n_routed, n_shared, top_k, expert_d_ff, aux_loss_coef=0.01):
        super().__init__()
        self.n_routed, self.top_k = n_routed, top_k
        self.aux_loss_coef = aux_loss_coef
        self.router = nn.Linear(d_model, n_routed, bias=False)
        self.routed_experts = nn.ModuleList([Expert(d_model, expert_d_ff) for _ in range(n_routed)])
        self.shared_experts = nn.ModuleList([Expert(d_model, expert_d_ff) for _ in range(n_shared)])
        self.last_aux_loss = None

    def forward(self, x):
        B, T, D = x.shape
        flat = x.reshape(-1, D)                                 # (N, D), N = B*T
        N = flat.shape[0]

        logits = self.router(flat)                               # (N, n_routed)
        probs = F.softmax(logits, dim=-1)
        top_w, top_i = probs.topk(self.top_k, dim=-1)             # (N, top_k) each
        top_w = top_w / top_w.sum(dim=-1, keepdim=True)           # renormalize among selected

        out = torch.zeros_like(flat)
        for e, expert in enumerate(self.routed_experts):
            token_slot, k_slot = (top_i == e).nonzero(as_tuple=True)   # which tokens picked expert e, and at which of their top_k slots
            if token_slot.numel() == 0:
                continue
            expert_out = expert(flat[token_slot])                       # only THIS expert's assigned tokens, real compute saving
            out.index_add_(0, token_slot, expert_out * top_w[token_slot, k_slot].unsqueeze(-1))

        for expert in self.shared_experts:
            out = out + expert(flat)                                    # always on, every token

        if self.training:
            # Switch-Transformer style load-balancing aux loss: encourages the router
            # to spread tokens evenly across experts rather than collapsing onto a few.
            with torch.no_grad():
                assign_count = torch.zeros(self.n_routed, device=x.device)
                assign_count.scatter_add_(0, top_i.reshape(-1), torch.ones_like(top_i.reshape(-1), dtype=torch.float))
                frac_tokens = assign_count / (N * self.top_k)
            frac_prob = probs.mean(dim=0)
            self.last_aux_loss = self.aux_loss_coef * self.n_routed * (frac_tokens * frac_prob).sum()
        else:
            self.last_aux_loss = None

        return out.view(B, T, D)

In [ ]:
%%writefile /kaggle/working/mla_dsa.py
import torch
import torch.nn as nn
import torch.nn.functional as F

from mla import MLA
from dsa import Indexer, chunked_sparse_attention, indexer_distillation_loss


class MLADSA(nn.Module):
    def __init__(self, d_model, n_head, head_dim, d_c, d_c_q, d_rotate,
                 n_idx_heads, idx_head_dim, chunk_size, topk_chunks, forced_local_chunks,
                 q_block=128, indexer_distill_sample=256, dropout=0.0, rope_theta=10000.0):
        super().__init__()
        self.mla = MLA(d_model, n_head, head_dim, d_c, d_c_q, d_rotate, dropout, rope_theta)
        self.indexer = Indexer(d_c_q, d_c, n_idx_heads, idx_head_dim, chunk_size, rope_theta)
        self.chunk_size, self.topk_chunks = chunk_size, topk_chunks
        self.forced_local_chunks, self.q_block = forced_local_chunks, q_block
        self.indexer_distill_sample = indexer_distill_sample

    def forward(self, x, past_kv=None, use_cache=False, sparse=True):
        B, T, _ = x.shape
        past_len = past_kv[0].shape[1] if past_kv is not None else 0

        c_kv, c_q, k_R = self.mla.project(x)
        if past_kv is not None:
            full_c_kv = torch.cat([past_kv[0], c_kv], dim=1)
            full_k_R = torch.cat([past_kv[1], k_R], dim=1)
        else:
            full_c_kv, full_k_R = c_kv, k_R

        Q, _, _ = self.mla.up_project(c_kv, c_q, k_R, past_len, x.device, x.dtype)
        _, K_full, V_full = self.mla._up_project_kv_only(full_c_kv, full_k_R, x.device, x.dtype)

        self.last_indexer_loss = None
        if sparse:
            idx_scores = self.indexer(c_q, full_c_kv, past_len)          # (B, T, past_len+T)
            out = chunked_sparse_attention(
                Q, K_full, V_full, idx_scores, self.chunk_size, self.topk_chunks,
                self.forced_local_chunks, self.q_block, causal_offset=past_len,
            )
            if self.training:
                self.last_indexer_loss = indexer_distillation_loss(
                    Q, K_full, idx_scores, self.chunk_size, causal_offset=past_len,
                    sample_size=self.indexer_distill_sample,
                )
        else:
            total_len = past_len + T
            q_pos = torch.arange(past_len, total_len, device=x.device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=x.device).unsqueeze(0)
            # Q.dtype, not x.dtype -- same fix as mla.py, same reason (x is pre-projection
            # and can be fp32 under autocast even when Q is correctly fp16 post-projection)
            bias = (k_pos > q_pos).to(Q.dtype) * torch.finfo(Q.dtype).min
            out = F.scaled_dot_product_attention(Q, K_full, V_full, attn_mask=bias[None, None])

        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        out = self.mla.W_O(out)
        present = (full_c_kv, full_k_R) if use_cache else None
        return out, present

In [ ]:
%%writefile /kaggle/working/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

from mla_dsa import MLADSA
from moe import MoE


class LegalMLADSAConfig(PretrainedConfig):
    model_type = "legal_mla_dsa"

    def __init__(
        self,
        vocab_size=50257, n_positions=4096, n_embd=1024, n_layer=8, n_head=16, head_dim=64,
        d_c=128, d_c_q=192, d_rotate=32,
        n_idx_heads=8, idx_head_dim=32, chunk_size=16, topk_chunks=64, forced_local_chunks=2, q_block=128,
        indexer_distill_sample=256,
        n_routed_experts=6, n_shared_experts=1, moe_top_k=2, expert_d_ff=896, moe_aux_coef=0.01,
        indexer_loss_coef=0.1,
        dropout=0.0, initializer_range=0.02, rms_norm_eps=1e-5,
        bos_token_id=50256, eos_token_id=50256, pad_token_id=50256,
        activation_checkpointing=False, tie_word_embeddings=True,
        **kwargs,
    ):
        self.vocab_size, self.n_positions, self.n_embd = vocab_size, n_positions, n_embd
        self.n_layer, self.n_head, self.head_dim = n_layer, n_head, head_dim
        self.d_c, self.d_c_q, self.d_rotate = d_c, d_c_q, d_rotate
        self.n_idx_heads, self.idx_head_dim = n_idx_heads, idx_head_dim
        self.chunk_size, self.topk_chunks, self.forced_local_chunks, self.q_block = \
            chunk_size, topk_chunks, forced_local_chunks, q_block
        self.indexer_distill_sample = indexer_distill_sample
        self.n_routed_experts, self.n_shared_experts, self.moe_top_k, self.expert_d_ff = \
            n_routed_experts, n_shared_experts, moe_top_k, expert_d_ff
        self.moe_aux_coef, self.indexer_loss_coef = moe_aux_coef, indexer_loss_coef
        self.dropout, self.initializer_range, self.rms_norm_eps = dropout, initializer_range, rms_norm_eps
        self.use_cache = True
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id, eos_token_id=eos_token_id, pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings, **kwargs,
        )


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class DecoderBlock(nn.Module):
    def __init__(self, config: LegalMLADSAConfig):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = MLADSA(
            config.n_embd, config.n_head, config.head_dim, config.d_c, config.d_c_q, config.d_rotate,
            config.n_idx_heads, config.idx_head_dim, config.chunk_size, config.topk_chunks,
            config.forced_local_chunks, config.q_block, config.indexer_distill_sample, config.dropout,
        )
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.moe = MoE(config.n_embd, config.n_routed_experts, config.n_shared_experts,
                        config.moe_top_k, config.expert_d_ff, config.moe_aux_coef)

    def forward(self, x, past_kv=None, use_cache=False, sparse=True):
        attn_out, present = self.attn(self.ln_1(x), past_kv, use_cache, sparse)
        x = x + attn_out
        x = x + self.moe(self.ln_2(x))
        return x, present, self.attn.last_indexer_loss, self.moe.last_aux_loss


class LegalMLADSAForCausalLM(PreTrainedModel):
    config_class = LegalMLADSAConfig
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalMLADSAConfig):
        super().__init__(config)
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        for name, p in module.named_parameters(recurse=False):
            if name in ("W_O.weight", "down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer))

    def get_input_embeddings(self):
        return self.wte

    def set_input_embeddings(self, value):
        self.wte = value

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def forward(self, input_ids, attention_mask=None, past_key_values=None, labels=None, use_cache=None):
        # fp16 autocast set HERE, not left to Trainer -- under nn.DataParallel the
        # matmuls measurably ran in fp32 otherwise (model 1's finding). Only for
        # unpadded batches; generation's padded path keeps exact fp32.
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)
        # NOTE: no **kwargs param above on purpose -- Trainer treats a forward()
        # that accepts **kwargs as self-normalizing its loss, then multiplies by
        # n_gpu and skips the grad-accumulation division (model 1's loss-scale bug).

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False
        past_len = past_key_values[0][0].shape[1] if past_key_values is not None else 0
        total_len = past_len + input_ids.shape[1]
        # decide sparse/dense from the TOTAL context (cache included), not just this
        # call's input length -- otherwise single-token generation steps always see
        # length 1 and silently fall back to dense every time, even with a long cache
        sparse = total_len > self.config.chunk_size * 2

        x = self.drop(self.wte(input_ids))
        presents = [] if use_cache else None
        indexer_losses, moe_losses = [], []
        for i, block in enumerate(self.blocks):
            pkv = past_key_values[i] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                def run(h, blk=block, pkv=pkv):
                    h2, _, idx_l, moe_l = blk(h, pkv, False, sparse)
                    if idx_l is not None:
                        indexer_losses.append(idx_l)
                    if moe_l is not None:
                        moe_losses.append(moe_l)
                    return h2
                x = torch_checkpoint(run, x, use_reentrant=False)
            else:
                x, present, idx_l, moe_l = block(x, pkv, use_cache, sparse)
                if use_cache:
                    presents.append(present)
                if idx_l is not None:
                    indexer_losses.append(idx_l)
                if moe_l is not None:
                    moe_losses.append(moe_l)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100,
            )
            if indexer_losses:
                loss = loss + self.config.indexer_loss_coef * torch.stack(indexer_losses).mean()
            if moe_losses:
                loss = loss + torch.stack(moe_losses).mean()

        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")
from model import LegalMLADSAConfig, LegalMLADSAForCausalLM

config = LegalMLADSAConfig(
    vocab_size=len(tokenizer), n_positions=BLOCK,
    n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, head_dim=HEAD_DIM,
    d_c=D_C, d_c_q=D_C_Q, d_rotate=D_ROTATE,
    n_idx_heads=N_IDX_HEADS, idx_head_dim=IDX_HEAD_DIM, chunk_size=CHUNK_SIZE,
    topk_chunks=TOPK_CHUNKS, forced_local_chunks=FORCED_LOCAL_CHUNKS, q_block=Q_BLOCK,
    n_routed_experts=N_ROUTED_EXPERTS, n_shared_experts=N_SHARED_EXPERTS,
    moe_top_k=MOE_TOP_K, expert_d_ff=EXPERT_D_FF, indexer_loss_coef=INDEXER_LOSS_COEF,
    indexer_distill_sample=INDEXER_DISTILL_SAMPLE,
    bos_token_id=tokenizer.bos_token_id, eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    activation_checkpointing=True,
)
model = LegalMLADSAForCausalLM(config)
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

## 5. Training Arguments

`max_steps` is the **whole** training length, so the cosine schedule is one continuous curve across sessions. `StopAfter` ends *this session* early and forces a checkpoint.

In [ ]:
from transformers import (TrainingArguments, Trainer, TrainerCallback,
                          DataCollatorForLanguageModeling)
from transformers.trainer_utils import get_last_checkpoint

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    max_steps=TOTAL_TRAIN_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH * 2,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.1,
    adam_beta1=0.9, adam_beta2=0.95,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    fp16=torch.cuda.is_available(),      # T4 is Turing: fp16, not bf16
    max_grad_norm=1.0,
    eval_strategy="steps", eval_steps=EVAL_STEPS, prediction_loss_only=True,
    logging_steps=10,
    save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=2,
    dataloader_num_workers=2,
    seed=SEED,
    report_to="none",
    disable_tqdm=True,   # the notebook progress widget has a bug that crashes a session ending at exactly 2 steps; plain-text logs suit a commit anyway
)

class StopAfter(TrainerCallback):
    """Ends THIS session (with a checkpoint) at a step count OR a wall-clock deadline."""
    def __init__(self, stop_step, deadline):
        self.stop_step, self.deadline = stop_step, deadline
    def on_step_end(self, args, state, control, **kwargs):
        out_of_time = time.time() > self.deadline
        if state.global_step >= self.stop_step or out_of_time:
            if out_of_time and state.global_step < self.stop_step:
                print(f"\nSession time limit reached at step {state.global_step}: checkpointing and stopping.")
            control.should_save = True
            control.should_training_stop = True
        return control

## 6. Train (auto-resume + per-step timing)

In [ ]:
# --- pull the newest checkpoint from a previously attached output, if we have none yet ---
if not glob.glob(os.path.join(MODEL_OUTPUT_DIR, "checkpoint-*")):
    prev = glob.glob("/kaggle/input/**/legal-mladsa-final/checkpoint-*", recursive=True)
    if prev:
        latest = max(prev, key=lambda p: int(p.rsplit("-", 1)[1]))
        shutil.copytree(latest, os.path.join(MODEL_OUTPUT_DIR, os.path.basename(latest)))
        prev_meta = os.path.join(os.path.dirname(os.path.dirname(latest)), "training_meta.json")
        if os.path.exists(prev_meta):
            shutil.copy(prev_meta, TRAINING_META_PATH)
        print("Restored checkpoint from previous output:", latest)

# --- refuse to resume into a differently-configured run ---
fingerprint = {"global_seqs": GLOBAL_SEQS, "block": BLOCK, "total_steps": TOTAL_TRAIN_STEPS,
               "arch": [N_EMBD, N_LAYER, N_HEAD, HEAD_DIM, D_C, D_C_Q, D_ROTATE,
                        N_IDX_HEADS, IDX_HEAD_DIM, CHUNK_SIZE, TOPK_CHUNKS, FORCED_LOCAL_CHUNKS, Q_BLOCK,
                        N_ROUTED_EXPERTS, N_SHARED_EXPERTS, MOE_TOP_K, EXPERT_D_FF, INDEXER_DISTILL_SAMPLE],
               "lr": LEARNING_RATE, "warmup": WARMUP_STEPS,
               "data": [TOKEN_BUDGET, str(MIX), SEED, VAL_RUNS, VAL_RUN_BLOCKS, VAL_MARGIN_BLOCKS]}
cumulative_seconds = 0.0
if os.path.exists(TRAINING_META_PATH):
    saved = json.load(open(TRAINING_META_PATH))
    cumulative_seconds = saved.get("cumulative_seconds", 0.0)
    if saved.get("fingerprint") != fingerprint and glob.glob(os.path.join(MODEL_OUTPUT_DIR, "checkpoint-*")):
        raise RuntimeError(f"Checkpoint was made with a different config!\n saved:   {saved.get('fingerprint')}\n current: {fingerprint}\n"
                           "Either restore the old settings or start fresh (don't attach the old output).")

last_checkpoint = get_last_checkpoint(MODEL_OUTPUT_DIR)
steps_at_start = int(last_checkpoint.rsplit("-", 1)[1]) if last_checkpoint else 0
stop_step = TOTAL_TRAIN_STEPS if STEPS_THIS_RUN is None else min(steps_at_start + STEPS_THIS_RUN, TOTAL_TRAIN_STEPS)
deadline = SESSION_START + MAX_SESSION_HOURS * 3600
print(f"Resuming from step {steps_at_start}" if last_checkpoint else "Starting fresh.")
print(f"This session runs to step {stop_step} of {TOTAL_TRAIN_STEPS} (or until {MAX_SESSION_HOURS}h of session time)")

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    data_collator=data_collator, processing_class=tokenizer,
    callbacks=[StopAfter(stop_step, deadline)],
)

# ---- diagnostics: is the GPU busy, and are matmuls really running in fp16? ----
import subprocess, threading
gpu_samples, probe = [], {}
def _num(v):
    try: return float(v)
    except ValueError: return float("nan")
def _monitor(stop):
    while not stop.is_set():
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,power.draw,clocks.sm",
                 "--format=csv,noheader,nounits"], text=True, timeout=10)
            gpu_samples.append([[_num(v.strip()) for v in ln.split(",")] for ln in out.strip().splitlines()])
        except Exception:
            return
        stop.wait(5)
_stop = threading.Event()
threading.Thread(target=_monitor, args=(_stop,), daemon=True).start()
def _record_dtype(module, inputs, output):
    probe.setdefault("W_UQ output dtype", str(output.dtype))   # returns None: must not replace the output
_hook = model.blocks[0].attn.mla.W_UQ.register_forward_hook(_record_dtype)

t0 = time.time()
trainer.train(resume_from_checkpoint=last_checkpoint)
elapsed = time.time() - t0

_stop.set(); _hook.remove()
print("\n--- diagnostics ---")
print("matmul output dtype inside the model:", probe.get("W_UQ output dtype", "n/a"), "(float16 = tensor cores in use)")
if gpu_samples:
    for g in range(len(gpu_samples[0])):
        col = lambda k: [smp[g][k] for smp in gpu_samples if len(smp) > g and smp[g][k] == smp[g][k]]
        m = lambda xs: sum(xs) / len(xs) if xs else float("nan")
        print(f"GPU {g}: util {m(col(1)):.0f}% | mem peak {max(col(2) or [0]):.0f} MB | power {m(col(3)):.0f} W | SM clock {m(col(4)):.0f} MHz  ({len(gpu_samples)} samples)")
else:
    print("nvidia-smi sampling unavailable")
if torch.cuda.is_available():
    print("torch peak allocated:", [f"{torch.cuda.max_memory_allocated(i)/2**30:.1f} GB" for i in range(torch.cuda.device_count())])

steps_done = trainer.state.global_step - steps_at_start
sec_per_step = elapsed / max(steps_done, 1)
cumulative_seconds += elapsed
json.dump({"cumulative_seconds": cumulative_seconds, "global_step": trainer.state.global_step,
           "fingerprint": fingerprint}, open(TRAINING_META_PATH, "w"))

remaining_steps = TOTAL_TRAIN_STEPS - trainer.state.global_step
print()
print(f"Steps this session: {steps_done} | wall time: {elapsed/60:.1f} min | {sec_per_step:.1f} s/step")
print(f"Progress: step {trainer.state.global_step:,} / {TOTAL_TRAIN_STEPS:,}")
print(f"GPU time used so far: {cumulative_seconds/3600:.2f} / {TOTAL_GPU_HOUR_BUDGET:.0f} h")
print(f"Full remaining training at this speed: {remaining_steps * sec_per_step / 3600:.1f} h")

# ---- planning helper: how much data can the whole run afford at this speed? ----
budget_steps = int(TRAIN_BUDGET_HOURS * 3600 / sec_per_step)
fit_tokens = budget_steps * GLOBAL_SEQS * BLOCK
print()
print(f"At {sec_per_step:.1f} s/step, {TRAIN_BUDGET_HOURS:.0f}h of training = {budget_steps:,} steps = {budget_steps * GLOBAL_SEQS * BLOCK / 1e9:.2f}B tokens")
print(f"  -> real run: set TOKEN_BUDGET = {int(fit_tokens // 10_000_000 * 10_000_000):,}  (the corpus has {N_BLOCKS * BLOCK / 1e9:.2f}B)")
print(f"  -> ~{math.ceil(TRAIN_BUDGET_HOURS / MAX_SESSION_HOURS)} sessions of up to {MAX_SESSION_HOURS:.0f}h; set STEPS_THIS_RUN = None and each one stops itself in time")
if steps_done <= 3:
    print("(few steps measured: first-step overhead inflates this; treat as an upper bound)")
print()
print("If this OOM'd instead of printing the above: halve PER_DEVICE_BATCH and double GRAD_ACCUM_STEPS, re-run.")

## 6b. Where Is The Time Going? (per-component breakdown)

Uses a single un-parallelized copy of the model on one GPU, not the `DataParallel`-wrapped training model -- DataParallel scatters a fresh replica to each GPU per call, so there's no single place to read accumulated per-component timings back from. This measures the RELATIVE cost of each piece, which is what actually matters for deciding where to look; it deliberately doesn't try to match the real multi-GPU step time.

In [ ]:
if torch.cuda.is_available():
    probe = LegalMLADSAForCausalLM(config).to("cuda:0").eval()
    state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
    probe.load_state_dict(state)

    loader = torch.utils.data.DataLoader(train_dataset, batch_size=PER_DEVICE_BATCH, collate_fn=data_collator)
    batch = next(iter(loader))
    input_ids = batch["input_ids"].to("cuda:0")

    def timed(fn, warmup=1, reps=3):
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(reps):
            result = fn()
        torch.cuda.synchronize()
        return (time.time() - t0) / reps, result

    times = {}
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
        times["embedding"], x = timed(lambda: probe.drop(probe.wte(input_ids)))

        for i, block in enumerate(probe.blocks):
            xn_time, xn = timed(lambda: block.ln_1(x))
            times.setdefault("ln_1", 0.0); times["ln_1"] += xn_time

            proj_time, (c_kv, c_q, k_R) = timed(lambda: block.attn.mla.project(xn))
            times.setdefault("mla_project", 0.0); times["mla_project"] += proj_time

            upq_time, (Q, _, _) = timed(lambda: block.attn.mla.up_project(c_kv, c_q, k_R, 0, xn.device, xn.dtype))
            times.setdefault("mla_up_project_Q", 0.0); times["mla_up_project_Q"] += upq_time

            upkv_time, (_, K_full, V_full) = timed(lambda: block.attn.mla._up_project_kv_only(c_kv, k_R, xn.device, xn.dtype))
            times.setdefault("mla_up_project_KV", 0.0); times["mla_up_project_KV"] += upkv_time

            idx_time, idx_scores = timed(lambda: block.attn.indexer(c_q, c_kv, 0))
            times.setdefault("dsa_indexer", 0.0); times["dsa_indexer"] += idx_time

            from dsa import chunked_sparse_attention
            attn_time, attn_out = timed(lambda: chunked_sparse_attention(
                Q, K_full, V_full, idx_scores, CHUNK_SIZE, TOPK_CHUNKS, FORCED_LOCAL_CHUNKS, Q_BLOCK))
            times.setdefault("dsa_chunked_attention", 0.0); times["dsa_chunked_attention"] += attn_time

            out_flat = attn_out.transpose(1, 2).contiguous().view(x.shape[0], x.shape[1], -1)
            wo_time, attn_final = timed(lambda: block.attn.mla.W_O(out_flat))
            times.setdefault("mla_output_proj", 0.0); times["mla_output_proj"] += wo_time

            x2 = x + attn_final
            ln2_time, xn2 = timed(lambda: block.ln_2(x2))
            times.setdefault("ln_2", 0.0); times["ln_2"] += ln2_time

            moe_time, moe_out = timed(lambda: block.moe(xn2))
            times.setdefault("moe", 0.0); times["moe"] += moe_time

            x = x2 + moe_out

    total = sum(times.values())
    print(f"Per-component time (one forward pass, {len(probe.blocks)} layers, single GPU, batch={PER_DEVICE_BATCH}):")
    for k, v in sorted(times.items(), key=lambda kv: -kv[1]):
        print(f"  {k:24s} {v*1000:8.1f} ms  ({100*v/total:5.1f}%)")
    print(f"  {'TOTAL':24s} {total*1000:8.1f} ms")
    del probe
    torch.cuda.empty_cache()
else:
    print("No GPU available -- skipping component timing breakdown.")

## 7. Validation Loss, Loss Plot, Save Model

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

eval_loss = trainer.evaluate()["eval_loss"]
print(f"Validation loss: {eval_loss:.4f} | perplexity: {math.exp(eval_loss):.2f}")

logs = trainer.state.log_history
tr = [(l["step"], l["loss"]) for l in logs if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]
plt.figure(figsize=(9, 4.5))
if tr: plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), label="val loss")
plt.xlabel("optimizer step"); plt.ylabel("loss"); plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss.png"), dpi=150)

print("Per-source validation:")
for s in SOURCES:
    r = trainer.evaluate(eval_dataset=val_by_source[s], metric_key_prefix=f"val_{s}")
    print(f"  {s:14s} loss {r[f'val_{s}_loss']:.4f} | perplexity {math.exp(r[f'val_{s}_loss']):.1f}")

trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print("Saved model + tokenizer to", MODEL_OUTPUT_DIR)